# Apurva — ML-Based Network Intrusion Detector

**Goal:** Train and evaluate a beginner-friendly classifier using a public labelled network-flow CSV.

**Dataset:** [Official CIC-IDS2017 dataset page](https://www.unb.ca/cic/datasets/ids-2017.html)

> Before running the training cells, download a labelled CSV file and save it as `data/network_traffic.csv`. Check the label column name first. Training is capped at 100,000 rows by default to keep it manageable on a typical laptop.

## 1. Install dependencies

Run in a terminal from the project folder:

```bash
pip install -r requirements.txt
```

## 2. Check the dataset columns

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/network_traffic.csv")
if not DATA_PATH.exists():
    print("Dataset not found. Download a labelled CSV from the official CIC-IDS2017 page and save it to:", DATA_PATH)
else:
    preview = pd.read_csv(DATA_PATH, nrows=5, low_memory=False)
    print("Rows preview:", len(preview))
    print("Columns:", preview.columns.tolist())
    display(preview.head())

## 3. Train and compare models

This calls the included training script, which handles preprocessing, compares Decision Tree and Random Forest, prints classification metrics, and saves the best pipeline and report charts.

In [ ]:
# If your target column is not named Label, change it below.
# Run from the project root. You can also use the terminal command in README.md.
import subprocess, sys

if DATA_PATH.exists():
    result = subprocess.run(
        [sys.executable, "src/train_model.py", "--data", str(DATA_PATH), "--target", "Label"],
        text=True, capture_output=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print("Add the dataset first, then rerun this cell.")

## 4. View evaluation results

After training, the script writes `reports/metrics.json`, `reports/confusion_matrix.png`, and (where supported) `reports/feature_importance.png`.

In [ ]:
import json
from IPython.display import display, Image

metrics_path = Path("reports/metrics.json")
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("Selected model:", metrics["selected_model"])
    for model_name, values in metrics["models"].items():
        print(f"\n{model_name}:")
        for metric in ("accuracy", "precision_macro", "recall_macro", "f1_macro"):
            print(f"  {metric}: {values[metric]:.4f}")
    if Path("reports/confusion_matrix.png").exists():
        display(Image(filename="reports/confusion_matrix.png"))
else:
    print("Train the model first.")

## 5. Launch the dashboard

In a terminal, from the project root:

```bash
streamlit run app.py
```

Upload a CSV that has the same input feature columns as the training dataset. The app will display predictions and let you download the results.

## Limitations

This is a learning/demo project, not a production security system. It predicts from CSV data; it does not monitor live network traffic or block attacks. Report actual measured results only after running the notebook/script.